# CIFAR Wilcoxon Test

Run the Wilcoxon test notebook after training a model. This notebook clones the repo into Colab if needed and runs the test logic directly in the notebook.

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/DVBW')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--branch', 'edge-amplify', 'https://github.com/THUYimingLi/DVBW.git', str(repo_dir)], check=True)
    print(f'Cloned repository to {repo_dir} from edge-amplify branch')
else:
    # Checkout edge-amplify branch if repo already exists
    subprocess.run(['git', '-C', str(repo_dir), 'checkout', 'edge-amplify'], check=True)
    print(f'Repository already present at {repo_dir}, checked out edge-amplify branch')

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), Path.cwd() / 'CIFAR', Path('/content/DVBW/CIFAR'), Path('/content/drive/MyDrive/DVBW/CIFAR')]
WORKDIR = next((path for path in candidates if (path / 'model.py').exists()), None)
if WORKDIR is None:
    raise FileNotFoundError('Could not find the CIFAR folder. Set WORKDIR manually if needed.')
os.chdir(WORKDIR)
print(f'Working directory: {WORKDIR}')

In [ ]:
%pip install -q scipy pillow numpy

In [ ]:
MODEL = 'resnet'  # 'resnet' or 'vgg'
MODEL_PATH = './checkpoint/infected/resnet_badnets_edge-amp/checkpoint.pth.tar'
CLEAN_MODEL_PATH = './checkpoint/benign/resnet/checkpoint.pth.tar'
TRIGGER_PATH = './triggers/Trigger_cross.png'
ALPHA_PATH = './triggers/Alpha_cross.png'
TARGET_LABEL = 0
NUM_IMG = 100
TEST_BATCH = 16
WORKERS = 2
GPU_ID = '0'
SEED = 666

In [ ]:
import os
import random

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from scipy.stats import wilcoxon

from model import *
from tools import *

assert MODEL in {'resnet', 'vgg'}
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
use_cuda = torch.cuda.is_available()
if not use_cuda:
    raise RuntimeError('CUDA GPU not available. In Colab, enable a GPU runtime and rerun.')

data_dir = WORKDIR / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
datasets.CIFAR10(root=str(data_dir), train=False, download=True)

# Create edge-amplify transforms (deterministic, no trigger images needed)
trigger = torch.zeros([3, 32, 32])
alpha = torch.zeros([3, 32, 32], dtype=torch.float)
edge_amplify = TriggerAppending(trigger=trigger, alpha=alpha)
edge_amplify.watermark = 'edge_amplify'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f'Using model: {MODEL_PATH}')
print(f'Using clean baseline: {CLEAN_MODEL_PATH}')


def build_model(model_name):
    if model_name == 'resnet':
        return ResNet18()
    return vgg19_bn()


def collect_argmax_outputs(testloader, model):
    model.eval()
    outputs_all = []
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.cuda(), targets.cuda()
            outputs = model(inputs)
            outputs_all += torch.argmax(outputs, dim=1).cpu().numpy().tolist()
    return np.array(outputs_all)


def main():
    main_model = build_model(MODEL)
    clean_model = build_model(MODEL)

    main_checkpoint = torch.load(MODEL_PATH)
    clean_checkpoint = torch.load(CLEAN_MODEL_PATH)

    main_model = torch.nn.DataParallel(main_model).cuda()
    clean_model = torch.nn.DataParallel(clean_model).cuda()
    main_model.load_state_dict(main_checkpoint['state_dict'])
    clean_model.load_state_dict(clean_checkpoint['state_dict'])
    main_model.eval()
    clean_model.eval()
    cudnn.benchmark = True

    transform_test_poisoned = transforms.Compose([edge_amplify, transforms.ToTensor()])

    dataloader = datasets.CIFAR10
    test_set_basic = dataloader(root=str(data_dir), train=False, download=True)
    testset_poisoned = dataloader(root=str(data_dir), train=False, download=True, transform=transform_test_poisoned)

    select_img = []
    select_target = []
    for i in range(len(test_set_basic)):
        if test_set_basic.targets[i] != TARGET_LABEL:
            select_img.append(test_set_basic.data[i])
            select_target.append(test_set_basic.targets[i])

    idx = list(np.arange(len(select_img)))
    random.shuffle(idx)
    image_idx = idx[:NUM_IMG]

    testing_img_poisoned = [select_img[i] for i in range(len(select_img)) if i in image_idx]
    testing_target = [select_target[i] for i in range(len(select_img)) if i in image_idx]

    testset_poisoned.data, testset_poisoned.targets = testing_img_poisoned, testing_target

    poisoned_loader = torch.utils.data.DataLoader(testset_poisoned, batch_size=TEST_BATCH, shuffle=False, num_workers=WORKERS)

    output_main_poisoned = collect_argmax_outputs(poisoned_loader, main_model)
    output_clean_poisoned = collect_argmax_outputs(poisoned_loader, clean_model)

    w_malicious = wilcoxon(x=output_main_poisoned - TARGET_LABEL, zero_method='zsplit', alternative='two-sided', mode='approx')
    w_model_independent = wilcoxon(x=output_clean_poisoned - TARGET_LABEL, zero_method='zsplit', alternative='two-sided', mode='approx')

    path_folder = str(Path(MODEL_PATH).parent)
    print(f'Malicious Wtest p-value: {1 - w_malicious[1]:.4e}')
    print(f'Model Independent Wtest p-value: {1 - w_model_independent[1]:.4e}')

    output_path = Path(path_folder) / f'Wtest_{NUM_IMG}.txt'
    with open(output_path, 'w') as f:
        for i in range(len(output_main_poisoned)):
            f.write('{:04d} {:d} {:d}\n'.format(image_idx[i], TARGET_LABEL, output_main_poisoned[i]))
        f.write(f'Malicious Wtest p-value: {1 - w_malicious[1]:.4e}\n')
        f.write(f'Model Independent Wtest p-value: {1 - w_model_independent[1]:.4e}\n')
    print(f'Saved results to {output_path}')


main()